# MizanIQ Phase-2 OCR/Vision Benchmark (Kaggle lab)

First experiment: **A** Tesseract CPU baseline (DEV-004, DEV-010), then **B** PaddleOCR-VL-1.6
(DEV-004/010, optionally 008/009). **C** Qwen3-VL on DEV-008/009 is OPTIONAL and OFF by default.

**Setup:** upload the export bundle (`inputs/`, `manifest.json`, `ocr_benchmark_truth.json`,
`PROMPTS.md`, `RESULT_SCHEMA.json`) — see `notebooks/README.md`. CPU suffices for A;
enable GPU (T4) for B. Experiment B is fully runnable: `PaddleOCRVL(pipeline_version='v1.6')`.
No weights or secrets are stored in this notebook.
Rule: never hand-edit model outputs before scoring.

In [ ]:
# -- 0. Environment information --
import json
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

print('python:', platform.python_version(), '| impl:', platform.python_implementation())
print('platform:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch: not installed (CPU-only session)')
print('nvidia-smi:', shutil.which('nvidia-smi') or 'absent')
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True,
                         text=True).stdout.strip())

DEVICE = 'kaggle-GPU' if shutil.which('nvidia-smi') else 'kaggle-CPU'
print('DEVICE =', DEVICE)

## Setup - install PaddleOCR-VL-1.6 (KAGGLE ONLY, never run locally)

1. Read the environment report FIRST (CUDA/GPU decides the wheel).
2. Install SEPARATELY: PaddlePaddle GPU `>=3.2.1` (CUDA-matched per the official table
   (https://www.paddlepaddle.org.cn/install/quick), check CUDA via `nvcc --version`),
   then `paddleocr[doc-parser]` (doc-parser extra required for `PaddleOCRVL`).
3. Keep install and imports SEPARATE: run install, RESTART the kernel if Paddle was
   installed/updated (restart may be required), then run imports. If `import paddle`
   is CPU-only on GPU, re-check the CUDA-matched wheel.
4. FIRST-RUN DOWNLOAD: first `PaddleOCRVL(pipeline_version='v1.6')` downloads models
   (internet once). Init time (`init_latency_ms`) is NEVER OCR latency.


In [ ]:
# KAGGLE ONLY - STEP 1/2: install (do not import here).
# Run this cell, then RESTART the kernel if Paddle was installed/updated,
# then run the next cell (imports). Never combine install + import in one run.
print('cuda runtime (nvcc):',
      shutil.which('nvcc') and subprocess.run(
          ['nvcc', '--version'], capture_output=True, text=True
      ).stdout.strip().splitlines()[-1] or 'nvcc absent')

# PaddlePaddle GPU >=3.2.1 (CUDA-matched; see official table) + doc-parser pipeline.
%pip install -q paddlepaddle-gpu>=3.2.1
%pip install -q paddleocr[doc-parser]
print('installed: restart kernel if Paddle updated')


In [ ]:
# KAGGLE ONLY - STEP 2/2: imports (run AFTER install + optional kernel restart).
import importlib
print('paddleocr installed:', importlib.util.find_spec('paddleocr') is not None)
try:
    import paddleocr
    print('paddleocr:', getattr(paddleocr, '__version__', 'unknown'))
except ImportError as exc:
    print('paddleocr import failed:', exc)
    print('Restarted kernel? Re-run install, '
          'restart, then re-run this cell.')
try:
    import paddle
    print('paddle:', paddle.__version__)
except ImportError as exc:
    print('paddle import failed:', exc)
    print('If on GPU without CUDA paddle, install the CUDA-matched paddlepaddle-gpu '
          'wheel per the official table, then restart the kernel and re-run this cell.')


## 1. Load benchmark

Validate the manifest and show the selected DEV IDs. Truth is loaded for field-name
alignment and local self-checks only - model outputs are never edited to match it.

In [ ]:
WORKSPACE = Path('.')
manifest = json.loads((WORKSPACE / 'manifest.json').read_text(encoding='utf-8'))
truth = json.loads((WORKSPACE / 'ocr_benchmark_truth.json').read_text(encoding='utf-8'))

print(f"{'document':10} {'role':10} {'tasks':14} file")
for entry in manifest['entries']:
    path = WORKSPACE / entry['workspace_file']
    assert path.is_file(), f'missing input: {path}'
    print(f"{entry['document_id']:10} {entry['role']:10} "
          f"{','.join(entry['tasks']):14} {entry['workspace_file']}")
print('\nselected A/B:', ['DEV-004', 'DEV-010'], '| optional B/C:', ['DEV-008', 'DEV-009'])

results = []  # standard-schema result dicts accumulate here

## 2. Common result utilities

Mirrors `src/ocrbench/schema.py` (the local validator is authoritative).

In [ ]:
PROMPTS = {
    'ocr': ('Transcribe all visible text faithfully, top to bottom. '
            'Preserve numbers, identifiers, and punctuation exactly as shown. '
            'For Arabic text, output the logical reading order. '
            'Do not infer, translate, or fill in missing values.'),
    'table': ('Recover each visible table as structured rows. '
              'Associate every row label with its visible numeric value exactly '
              'as printed. Preserve numbers and identifiers character-for-character. '
              'Do not calculate, round, or invent missing cells. '
              'If a cell is unreadable, write UNREADABLE for that cell only.'),
    'chart': ('Step 1 - report only what is visible: chart title, axis labels, '
              'every labeled value, and the legend. '
              'Step 2 - separately, describe the trend using only the visible '
              'values (direction, notable shifts, highest/lowest point). '
              'Do not estimate values that are not labeled.'),
    'kpi': ('Recover each displayed KPI card label and its value exactly as shown, '
            'then each bar label and value, then the budget panel figures. '
            'Separately, quote the visible budget-status statement verbatim. '
            'Do not recompute variances or reinterpret the verdict.'),
}


def blank_result(model, model_version, document_id):
    return {'model': model, 'model_version': model_version, 'device': DEVICE,
            'document_id': document_id, 'latency_ms': 0.0, 'text': '',
            'fields': {}, 'tables': [], 'visual_description': '', 'warnings': []}


def check_result(result):
    required = {'model': str, 'model_version': str, 'device': str,
                'document_id': str, 'latency_ms': (int, float), 'text': str,
                'fields': dict, 'tables': list, 'visual_description': str,
                'warnings': list}
    errors = [f'missing: {k}' for k in required if k not in result]
    errors += [f'type: {k}' for k, t in required.items()
               if k in result and not isinstance(result[k], t)]
    return errors

## 3. Experiment A - Tesseract baseline (DEV-004, DEV-010)

CPU only. Uses the `tesseract` CLI directly (no wrapper dependency). If Tesseract is
unavailable, results record the fact and the notebook continues - no unsafe workarounds.

In [ ]:
TESS_DOCS = ['DEV-004', 'DEV-010']
TESS_LANG = 'ara+eng'  # fallback to 'eng' below if the ara pack is absent


def tesseract_version():
    try:
        out = subprocess.run(['tesseract', '--version'], capture_output=True,
                             text=True, timeout=30).stdout
        return out.splitlines()[0] if out else 'unknown'
    except Exception:
        return 'unavailable'


def tesseract_langs():
    try:
        out = subprocess.run(['tesseract', '--list-langs'], capture_output=True,
                             text=True, timeout=30).stdout
        return [line.strip() for line in out.splitlines()[1:] if line.strip()]
    except Exception:
        return []


TESS_AVAILABLE = shutil.which('tesseract') is not None
TESS_VERSION = tesseract_version() if TESS_AVAILABLE else 'unavailable'
langs = tesseract_langs() if TESS_AVAILABLE else []
lang = TESS_LANG if all(p in langs for p in ('ara', 'eng')) else 'eng'
print('tesseract:', TESS_VERSION, '| langs:', langs or 'n/a', '| using:', lang)

for doc_id in TESS_DOCS:
    entry = next(e for e in manifest['entries'] if e['document_id'] == doc_id)
    result = blank_result('tesseract', TESS_VERSION, doc_id)
    if not TESS_AVAILABLE:
        result['warnings'] = ['tesseract binary unavailable in this environment']
    else:
        if lang != TESS_LANG:
            result['warnings'] = [f'ara traineddata missing; fell back to {lang}']
        started = time.perf_counter()
        try:
            proc = subprocess.run(
                ['tesseract', str(WORKSPACE / entry['workspace_file']),
                 'stdout', '-l', lang],
                capture_output=True, text=True, timeout=600)
            result['text'] = proc.stdout
            if proc.returncode != 0:
                result['warnings'].append(f'tesseract stderr: {proc.stderr[:300]}')
        except Exception as exc:
            result['warnings'].append(f'tesseract failed: {type(exc).__name__}')
        result['latency_ms'] = (time.perf_counter() - started) * 1000.0
    result['warnings'].append(f'lang config: {lang}; output NOT hand-corrected')
    assert not check_result(result), check_result(result)
    results.append(result)
    print(doc_id, f"{result['latency_ms']:.0f} ms", f'{len(result["text"])} chars',
          result['warnings'])

## 4. Experiment B - PaddleOCR-VL-1.6 (DEV-004, DEV-010, optionally 008/009)

Authoritative API: `from paddleocr import PaddleOCRVL`, pipeline
`PaddleOCRVL(pipeline_version='v1.6')` - instantiated ONCE per session. No version
substitution: any incompatibility is recorded as a blocker, never worked around by
silently using another pipeline.

Pipeline options: official v1.6 defaults, NO overrides (orientation classification,
unwarping, layout detection, chart recognition all stay as the pipeline ships them -
layout detection therefore runs iff it is default pipeline behavior). First fair
baseline: nothing tuned to our fixtures; every non-default option would be recorded.

Latency split: model init/download (`init_latency_ms`, first run downloads official
model files - internet required once) is measured separately and NEVER counted as
per-document OCR latency. Raw structured output is preserved per document
(`paddle_raw_<DEV>.json`) alongside the normalized schema result.

In [ ]:
PADDLE_MODEL_ID = 'PaddleOCR-VL-1.6'
PADDLE_PIPELINE_VERSION = 'v1.6'
PADDLE_DOCS = ['DEV-004', 'DEV-010']
PADDLE_DOCS_OPTIONAL = ['DEV-008', 'DEV-009']
RUN_PADDLE_OPTIONAL = False
PADDLE_PIPELINE_OPTIONS = {
    'doc_orientation_classify': 'official-default (not overridden)',
    'doc_unwarping': 'official-default (not overridden)',
    'layout_detection': 'official-default (not overridden)',
    'chart_recognition': 'official-default (not overridden)',
}
_PADDLE_STATE = {'pipeline': None, 'init_latency_ms': None, 'identity': None}


def _paddle_identity():
    try:
        import paddleocr
        poc = getattr(paddleocr, '__version__', 'unknown')
    except ImportError:
        poc = 'not-installed'
    try:
        import paddle
        pdv = getattr(paddle, '__version__', 'unknown')
    except ImportError:
        pdv = 'not-installed'
    return {'candidate': 'paddleocr-vl', 'model_id': PADDLE_MODEL_ID,
            'pipeline_version': PADDLE_PIPELINE_VERSION,
            'paddleocr_version': poc, 'paddle_version': pdv,
            'device': DEVICE, 'pipeline_options': dict(PADDLE_PIPELINE_OPTIONS)}


def _get_paddle_pipeline():
    if _PADDLE_STATE['pipeline'] is None:
        from paddleocr import PaddleOCRVL
        started = time.perf_counter()
        _PADDLE_STATE['pipeline'] = PaddleOCRVL(
            pipeline_version=PADDLE_PIPELINE_VERSION)
        _PADDLE_STATE['init_latency_ms'] = (time.perf_counter() - started) * 1000.0
    return _PADDLE_STATE['pipeline']


# v1.6: parsing_res_list; HTML->grids. Mirrors paddle_adapter. No invented fields.
from html.parser import HTMLParser


class _TParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.tables, self._ts, self._rs, self._cs = [], [], [], []

    def handle_starttag(self, tag, attrs):
        t = tag.lower()
        if t == 'table':
            self._ts.append([])
        elif t == 'tr':
            if not self._ts:
                self._ts.append([])
            self._rs.append([])
        elif t in ('td', 'th'):
            if not self._rs:
                if not self._ts:
                    self._ts.append([])
                self._rs.append([])
            cs = 1
            for k, v in attrs or []:
                if k.lower() == 'colspan':
                    try:
                        cs = max(1, int(str(v).strip()))
                    except (ValueError, TypeError):
                        cs = 1
            self._cs.append([[], cs])
        elif t == 'br' and self._cs:
            self._cs[-1][0].append(' ')

    def handle_data(self, data):
        if self._cs:
            self._cs[-1][0].append(data)

    def handle_endtag(self, tag):
        t = tag.lower()
        if t in ('td', 'th'):
            if not self._cs:
                return
            ch, cs = self._cs.pop()
            self._rs[-1].append(''.join(ch).strip())
            for _ in range(cs - 1):
                self._rs[-1].append('')
        elif t == 'tr':
            r = self._rs.pop()
            if r:
                self._ts[-1].append(r)
        elif t == 'table':
            tbl = self._ts.pop()
            if tbl:
                self.tables.append(tbl)


def _html_tables(s):
    if not isinstance(s, str) or not s.strip():
        return []
    low = s.lower()
    if '<td' not in low and '<th' not in low and '<tr' not in low:
        return []
    p = _TParser()
    try:
        p.feed(s)
        p.close()
    except Exception:
        return []
    while p._cs:
        ch, cs = p._cs.pop()
        t = ''.join(ch).strip()
        if not p._rs:
            p._rs.append([])
        p._rs[-1].append(t)
        for _ in range(cs - 1):
            p._rs[-1].append('')
    while p._rs:
        r = p._rs.pop()
        if r:
            if not p._ts:
                p._ts.append([])
            p._ts[-1].append(r)
    while p._ts:
        tbl = p._ts.pop()
        if tbl:
            p.tables.append(tbl)
    return [t for t in p.tables if t and any(any(c.strip() for c in r) for r in t)]


def _page_json(o):
    if not isinstance(o, (str, bytes, dict, list, tuple)) and hasattr(o, 'json'):
        try:
            v = o.json() if callable(getattr(o, 'json')) else o.json
            if isinstance(v, dict):
                return v
        except Exception:
            return None
    if isinstance(o, dict):
        r = o.get('res')
        if isinstance(r, dict) and isinstance(r.get('parsing_res_list'), list):
            return o
        if isinstance(o.get('parsing_res_list'), list):
            return o
    return None


def _paddle_adapt(raw):
    texts, fields, tables, visuals, notes = [], {}, [], [], []
    pages = None
    if _page_json(raw) is not None:
        pages = [raw]
    elif isinstance(raw, (list, tuple)) and raw and all(_page_json(e) for e in raw):
        pages = list(raw)
    if pages is None:  # minimal legacy fallback (Paddle path is v1.6)
        if isinstance(raw, str):
            return raw, fields, tables, '', ['shape: plain string']
        if isinstance(raw, dict) and isinstance(raw.get('rec_texts'), list):
            return '\n'.join(str(t) for t in raw['rec_texts']), fields, tables, '', ['used rec_texts']
        return str(raw), fields, tables, '', ['shape: unrecognized; stringified']
    notes.append(f'v1.6 pages x{len(pages)}; src=parsing_res_list (not markdown)')
    for pi, pg in enumerate(pages):
        pj = _page_json(pg)
        blk = pj['res']['parsing_res_list'] if 'res' in pj else pj['parsing_res_list']
        notes.append(f'page {pi}: parsing_res_list x{len(blk)}')
        for b in blk:
            if not isinstance(b, dict):
                texts.append(str(b))
                continue
            lab = b.get('block_label', b.get('label', b.get('type', '')))
            c = b.get('block_content', b.get('content', b.get('text', '')))
            if lab == 'image':
                if isinstance(c, str) and c.strip():
                    visuals.append(c)
                    notes.append(f'page {pi}: image as visual, not text')
                else:
                    notes.append(f'page {pi}: ignored empty image block')
                continue
            if lab == 'table':
                if isinstance(c, str) and c.strip():
                    g = _html_tables(c)
                    if g:
                        for t in g:
                            tables.append(t)
                            for r in t:
                                texts.append(' | '.join(r))
                        notes.append(f"page {pi}: table rows={sum(len(t) for t in g)}")
                    else:
                        texts.append(c)
                continue
            if isinstance(c, str):
                if c.strip():
                    texts.append(c)
            elif c not in (None, ''):
                texts.append(str(c))
    notes.append('fields: none from v1.6 (semantic extraction downstream)')
    return '\n'.join(texts), fields, tables, '\n'.join(visuals), notes


def _raw_json(raw):
    if _page_json(raw) is not None:
        return _page_json(raw)
    if isinstance(raw, (list, tuple)) and all(_page_json(e) for e in raw):
        return [_page_json(e) for e in raw]
    return raw


PADDLE_PKG_OK, PADDLE_BLOCKER = True, ''
try:
    _get_paddle_pipeline()
    _PADDLE_STATE['identity'] = _paddle_identity()
    print(f"paddle pipeline ready: {PADDLE_MODEL_ID} "
          f"(paddleocr=={_PADDLE_STATE['identity']['paddleocr_version']}, "
          f"paddle=={_PADDLE_STATE['identity']['paddle_version']})")
    print(f"init/download took {_PADDLE_STATE['init_latency_ms']:.0f} ms "
          f"(NOT counted as OCR latency)")
except ImportError as exc:
    PADDLE_PKG_OK, PADDLE_BLOCKER = False, f'PADDLE_BLOCKER: paddleocr not installed ({exc})'
    _PADDLE_STATE['identity'] = _paddle_identity()
    print(PADDLE_BLOCKER)
except Exception as exc:
    PADDLE_PKG_OK = False
    PADDLE_BLOCKER = (f'PADDLE_BLOCKER: v1.6 pipeline failed: '
                      f'{type(exc).__name__}: {exc}')
    _PADDLE_STATE['identity'] = _paddle_identity()
    print(PADDLE_BLOCKER)

for doc_id in PADDLE_DOCS + (PADDLE_DOCS_OPTIONAL if RUN_PADDLE_OPTIONAL else []):
    entry = next(e for e in manifest['entries'] if e['document_id'] == doc_id)
    result = blank_result('paddleocr-vl', PADDLE_MODEL_ID, doc_id)
    result['warnings'].append('pipeline_version=v1.6; options: official defaults (no overrides)')
    if not PADDLE_PKG_OK:
        result['warnings'].append(PADDLE_BLOCKER or 'PADDLE_BLOCKER: unavailable')
    else:
        started = time.perf_counter()
        try:
            raw = _get_paddle_pipeline().predict(
                str(WORKSPACE / entry['workspace_file']))
            raw_path = f'paddle_raw_{doc_id}.json'
            with open(raw_path, 'w', encoding='utf-8') as f:
                json.dump(_raw_json(raw), f, ensure_ascii=False, default=str)
            text, fields, tables, visual, notes = _paddle_adapt(raw)
            result.update({'text': text, 'fields': fields,
                           'tables': tables, 'visual_description': visual})
            result['warnings'].extend(notes + [f'raw preserved at: {raw_path}'])
        except Exception as exc:
            result['warnings'].append(
                f'inference failed: {type(exc).__name__}: {exc}')
        result['latency_ms'] = (time.perf_counter() - started) * 1000.0
    assert not check_result(result), check_result(result)
    results.append(result)
    print(doc_id, f"{result['latency_ms']:.0f} ms", result['warnings'])


## 5. Experiment C - Qwen3-VL (OPTIONAL, DEV-008/009 visual understanding)

NOT required for the first run. Prefer `Qwen/Qwen3-VL-8B-Instruct` when the GPU runs it
reliably; use the 4B fallback only when documented. Never change size mid-comparison
without recording it. Keep extraction (labels/values) and interpretation (trend/verdict)
in separate outputs per the chart/KPI prompts.

In [ ]:
RUN_EXPERIMENT_C = False  # flip to True for the visual-reasoning round
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'  # fallback (record!): Qwen/Qwen3-VL-4B-Instruct
QWEN_DOCS = ['DEV-008', 'DEV-009']

if not RUN_EXPERIMENT_C:
    print('Experiment C skipped (optional; set RUN_EXPERIMENT_C=True to run)')
else:
    try:
        import transformers
        print('transformers:', transformers.__version__)
    except ImportError:
        print('QWEN_BLOCKER: transformers not installed')
    # Implement Qwen3-VL-Instruct inference here from official docs:
    # load QWEN_MODEL_ID, apply the chart prompt to DEV-008 and the KPI
    # prompt to DEV-009 (PROMPTS above), record per-doc latency, append
    # blank_result('qwen3-vl', QWEN_MODEL_ID, doc_id) dicts to results.
    print('TODO: Qwen3-VL inference (see cell markdown + PROMPTS)')

## 6. Export

Write `kaggle_results.json` + `experiment_meta.json`. Both are validated against the
result contract before saving. Download both to local `data/benchmark/results/`.
Paddle init/download time is reported in meta, never mixed into OCR latency.

In [ ]:
errors = {r['document_id']: check_result(r) for r in results}
bad = {k: v for k, v in errors.items() if v}
assert not bad, bad
with open('kaggle_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
meta = {'models': sorted({(r['model'], r['model_version']) for r in results}),
        'device': DEVICE, 'n_results': len(results),
        'exported_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
paddle_state = globals().get('_PADDLE_STATE') or {}
if paddle_state.get('identity'):
    meta['paddle'] = {'identity': paddle_state['identity'],
                      'init_latency_ms': paddle_state.get('init_latency_ms')}
with open('experiment_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"{'document':10} {'model':14} {'ms':>9}  warnings")
for r in results:
    print(f"{r['document_id']:10} {r['model']:14} {r['latency_ms']:9.0f}  "
          f"{'; '.join(r['warnings'])}")
print('\nsaved kaggle_results.json + experiment_meta.json')